In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import tensorflow as tf

b = low -0.5
mono = high 0.5





In [ ]:
real_low = np.load('/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/b_Class_genomap.npy')

In [ ]:
real_high = np.load('/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/mono_Class_genomap.npy')

In [ ]:
generator = tf.keras.models.load_model('/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset/generator.h5')

In [ ]:
real_data = np.vstack([real_low, real_high ])
real_data = real_data.astype(np.float32)
#real_data = np.clip(real_data , -5 ,5)
_MIN = real_data.min()
_MAX = real_data.max()
real_data = -1 + 2*(real_data - _MIN)/(_MAX - _MIN)
real_data = (real_data + 1) * 127.5

In [ ]:
low_data = real_data[:real_low.shape[0],:,:,0]
low_data = low_data[np.random.choice(low_data.shape[0] , 100)]

In [ ]:
high_data = real_data[real_low.shape[0]:,:,:,0]
high_data = high_data[np.random.choice(high_data.shape[0] , 100)]

In [ ]:
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

In [ ]:
def generate(labels, generator , latent_dim = 128, num_features=1):
  _noise = tf.random.normal(shape=(len(labels), latent_dim))
  labels = np.array(labels).reshape((len(labels),num_features))

  fake_images = generator.predict([_noise, labels])
  return fake_images

In [ ]:
synthetic_high_images = generate([0.5]*100 , generator)
synthetic_high_images = synthetic_high_images [:,:,:,0]
synthetic_high_images = (synthetic_high_images + 1) * 127.5

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 155ms/step


In [ ]:
synthetic_low_images = generate([-0.5]*100 , generator)
synthetic_low_images = synthetic_low_images [:,:,:,0]
synthetic_low_images = (synthetic_low_images + 1) * 127.5

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step


In [ ]:
# Compute SSIM scores for each pair of images
def compute_ssim_matrix(images1, images2):
    ssim_matrix = np.zeros((images1.shape[0], images2.shape[0]))
    for i, img1 in enumerate(images1):
        for j, img2 in enumerate(images2):
            ssim_score = ssim(img1, img2, data_range=255, win_size=3)
            ssim_matrix[i, j] = ssim_score.item()
    return ssim_matrix

In [ ]:
ssim_real_high_vs_high = compute_ssim_matrix(high_data, high_data)
ssim_real_high_vs_syn_high = compute_ssim_matrix(high_data, synthetic_high_images)
ssim_real_low_vs_low = compute_ssim_matrix(low_data, low_data)
ssim_real_low_vs_syn_low = compute_ssim_matrix(low_data, synthetic_low_images)
ssim_real_high_vs_low = compute_ssim_matrix(high_data, low_data)
ssim_syn_high_vs_syn_low = compute_ssim_matrix(synthetic_high_images, synthetic_low_images)
ssim_syn_high_vs_syn_high = compute_ssim_matrix(synthetic_high_images, synthetic_high_images)
ssim_syn_low_vs_syn_low = compute_ssim_matrix(synthetic_low_images, synthetic_low_images)
ssim_real_high_vs_syn_low = compute_ssim_matrix(high_data, synthetic_low_images)
ssim_real_low_vs_syn_high = compute_ssim_matrix(low_data, synthetic_high_images)

In [ ]:
# Calculate average SSIM scores, masking out NaN values
def calculate_masked_mean(ssim_matrix):
    masked_matrix = np.ma.masked_invalid(ssim_matrix)
    return np.mean(masked_matrix)

# Print the average SSIM scores
print("Average SSIM Score (Real High vs Real High):", calculate_masked_mean(ssim_real_high_vs_high))
print("Average SSIM Score (Real High vs Synthetic High):", calculate_masked_mean(ssim_real_high_vs_syn_high))
print("Average SSIM Score (Real Low vs Real Low):", calculate_masked_mean(ssim_real_low_vs_low))
print("Average SSIM Score (Real Low vs Synthetic Low):", calculate_masked_mean(ssim_real_low_vs_syn_low))
print("Average SSIM Score (Real High vs Real Low):", calculate_masked_mean(ssim_real_high_vs_low))
print("Average SSIM Score (Synthetic High vs Synthetic Low):", calculate_masked_mean(ssim_syn_high_vs_syn_low))
print("Average SSIM Score (Synthetic High vs Synthetic High):", calculate_masked_mean(ssim_syn_high_vs_syn_high))
print("Average SSIM Score (Synthetic Low vs Synthetic Low):", calculate_masked_mean(ssim_syn_low_vs_syn_low))
print("Average SSIM Score (Real High vs Synthetic Low):", calculate_masked_mean(ssim_real_high_vs_syn_low))
print("Average SSIM Score (Real Low vs Synthetic High):", calculate_masked_mean(ssim_real_low_vs_syn_high))


# Function to calculate masked standard deviation
def calculate_masked_std(ssim_matrix):
    masked_matrix = np.ma.masked_invalid(ssim_matrix)
    return np.std(masked_matrix)

# Print the standard deviations for each SSIM comparison
print("Standard Deviation of SSIM Score (Real High vs Real High):", calculate_masked_std(ssim_real_high_vs_high))
print("Standard Deviation of SSIM Score (Real High vs Synthetic High):", calculate_masked_std(ssim_real_high_vs_syn_high))
print("Standard Deviation of SSIM Score (Real Low vs Real Low):", calculate_masked_std(ssim_real_low_vs_low))
print("Standard Deviation of SSIM Score (Real Low vs Synthetic Low):", calculate_masked_std(ssim_real_low_vs_syn_low))
print("Standard Deviation of SSIM Score (Real High vs Real Low):", calculate_masked_std(ssim_real_high_vs_low))
print("Standard Deviation of SSIM Score (Synthetic High vs Synthetic Low):", calculate_masked_std(ssim_syn_high_vs_syn_low))
print("Standard Deviation of SSIM Score (Synthetic High vs Synthetic High):", calculate_masked_std(ssim_syn_high_vs_syn_high))
print("Standard Deviation of SSIM Score (Synthetic Low vs Synthetic Low):", calculate_masked_std(ssim_syn_low_vs_syn_low))
print("Standard Deviation of SSIM Score (Real High vs Synthetic low):", calculate_masked_std(ssim_real_high_vs_syn_low))
print("Standard Deviation of SSIM Score (Real Low vs Synthetic High):", calculate_masked_std(ssim_real_low_vs_syn_high))

Average SSIM Score (Real High vs Real High): 0.5060783002161557
Average SSIM Score (Real High vs Synthetic High): 0.5020482906282023
Average SSIM Score (Real Low vs Real Low): 0.5927219532191637
Average SSIM Score (Real Low vs Synthetic Low): 0.5787459238105326
Average SSIM Score (Real High vs Real Low): 0.5392551795192532
Average SSIM Score (Synthetic High vs Synthetic Low): 0.5285586634052531
Average SSIM Score (Synthetic High vs Synthetic High): 0.5331374888016097
Average SSIM Score (Synthetic Low vs Synthetic Low): 0.5958291891669575
Average SSIM Score (Real High vs Synthetic Low): 0.5327333366197677
Average SSIM Score (Real Low vs Synthetic High): 0.541799515273018
Standard Deviation of SSIM Score (Real High vs Real High): 0.0688439237729489
Standard Deviation of SSIM Score (Real High vs Synthetic High): 0.04037977402529017
Standard Deviation of SSIM Score (Real Low vs Real Low): 0.08227231423797256
Standard Deviation of SSIM Score (Real Low vs Synthetic Low): 0.057947869640872174

In [ ]:
# PSNR Matrix Calculation Function
def compute_psnr_matrix(images1, images2):
    psnr_matrix = np.zeros((images1.shape[0], images2.shape[0]))
    for i, img1 in enumerate(images1):
        for j, img2 in enumerate(images2):
            psnr_score = psnr(img1, img2, data_range=255)
            psnr_matrix[i, j] = psnr_score
    return psnr_matrix

In [ ]:
# Compute PSNR matrices for all comparisons
psnr_real_high_vs_high = compute_psnr_matrix(high_data, high_data)
psnr_real_high_vs_syn_high = compute_psnr_matrix(high_data, synthetic_high_images)
psnr_real_low_vs_low = compute_psnr_matrix(low_data, low_data)
psnr_real_low_vs_syn_low = compute_psnr_matrix(low_data, synthetic_low_images)
psnr_real_high_vs_low = compute_psnr_matrix(high_data, low_data)
psnr_syn_high_vs_syn_low = compute_psnr_matrix(synthetic_high_images, synthetic_low_images)
psnr_syn_high_vs_syn_high = compute_psnr_matrix(synthetic_high_images, synthetic_high_images)
psnr_syn_low_vs_syn_low = compute_psnr_matrix(synthetic_low_images, synthetic_low_images)
psnr_real_high_vs_syn_low = compute_psnr_matrix(high_data, synthetic_low_images)
psnr_real_low_vs_syn_high = compute_psnr_matrix(low_data, synthetic_high_images)

/usr/local/lib/python3.11/dist-packages/skimage/metrics/simple_metrics.py:168: RuntimeWarning: divide by zero encountered in scalar divide
  return 10 * np.log10((data_range**2) / err)


In [ ]:
# Function to calculate the mean PSNR
def calculate_psnr_mean(psnr_matrix):
    masked_matrix = np.ma.masked_invalid(psnr_matrix)
    return np.mean(masked_matrix)

# Function to calculate the standard deviation of PSNR
def calculate_psnr_std(psnr_matrix):
    masked_matrix = np.ma.masked_invalid(psnr_matrix)
    return np.std(masked_matrix)

# Print Average PSNR Scores
print("Average PSNR Score (Real High vs Real High):", calculate_psnr_mean(psnr_real_high_vs_high))
print("Average PSNR Score (Real High vs Synthetic High):", calculate_psnr_mean(psnr_real_high_vs_syn_high))
print("Average PSNR Score (Real Low vs Real Low):", calculate_psnr_mean(psnr_real_low_vs_low))
print("Average PSNR Score (Real Low vs Synthetic Low):", calculate_psnr_mean(psnr_real_low_vs_syn_low))
print("Average PSNR Score (Real High vs Real Low):", calculate_psnr_mean(psnr_real_high_vs_low))
print("Average PSNR Score (Synthetic High vs Synthetic Low):", calculate_psnr_mean(psnr_syn_high_vs_syn_low))
print("Average PSNR Score (Synthetic High vs Synthetic High):", calculate_psnr_mean(psnr_syn_high_vs_syn_high))
print("Average PSNR Score (Synthetic Low vs Synthetic Low):", calculate_psnr_mean(psnr_syn_low_vs_syn_low))
print("Average PSNR Score (Real High vs Synthetic low):", calculate_psnr_mean(psnr_real_high_vs_syn_low))
print("Average PSNR Score (Real Low vs Synthetic High):", calculate_psnr_mean(psnr_real_low_vs_syn_high))

# Print Standard Deviations of PSNR
print("Standard Deviation of PSNR Score (Real High vs Real High):", calculate_psnr_std(psnr_real_high_vs_high))
print("Standard Deviation of PSNR Score (Real High vs Synthetic High):", calculate_psnr_std(psnr_real_high_vs_syn_high))
print("Standard Deviation of PSNR Score (Real Low vs Real Low):", calculate_psnr_std(psnr_real_low_vs_low))
print("Standard Deviation of PSNR Score (Real Low vs Synthetic Low):", calculate_psnr_std(psnr_real_low_vs_syn_low))
print("Standard Deviation of PSNR Score (Real High vs Real Low):", calculate_psnr_std(psnr_real_high_vs_low))
print("Standard Deviation of PSNR Score (Synthetic High vs Synthetic Low):", calculate_psnr_std(psnr_syn_high_vs_syn_low))
print("Standard Deviation of PSNR Score (Synthetic High vs Synthetic High):", calculate_psnr_std(psnr_syn_high_vs_syn_high))
print("Standard Deviation of PSNR Score (Synthetic Low vs Synthetic Low):", calculate_psnr_std(psnr_syn_low_vs_syn_low))
print("Standard Deviation of PSNR Score (Real High vs Synthetic low):", calculate_psnr_std(psnr_real_high_vs_syn_low))
print("Standard Deviation of PSNR Score (Real Low vs Synthetic High):", calculate_psnr_std(psnr_real_low_vs_syn_high))

Average PSNR Score (Real High vs Real High): 28.49562205518225
Average PSNR Score (Real High vs Synthetic High): 28.237753531278354
Average PSNR Score (Real Low vs Real Low): 28.57335671295812
Average PSNR Score (Real Low vs Synthetic Low): 28.363310275722476
Average PSNR Score (Real High vs Real Low): 28.536738156529168
Average PSNR Score (Synthetic High vs Synthetic Low): 28.02337146873484
Average PSNR Score (Synthetic High vs Synthetic High): 28.183158076727477
Average PSNR Score (Synthetic Low vs Synthetic Low): 28.344828977772803
Average PSNR Score (Real High vs Synthetic low): 28.332727419636615
Average PSNR Score (Real Low vs Synthetic High): 28.26893692313976
Standard Deviation of PSNR Score (Real High vs Real High): 0.9094807039486047
Standard Deviation of PSNR Score (Real High vs Synthetic High): 0.8378920837521103
Standard Deviation of PSNR Score (Real Low vs Real Low): 1.561066654274034
Standard Deviation of PSNR Score (Real Low vs Synthetic Low): 1.2877379563221754
Standar